# Notebook 05: Dynamic analysis, evasion, honest comparison and the full cascade

**Class 04_02, demos D5-D8.** (D5) What a sandbox-aware sample does to the *trace* and what the model sees;
(D6) machine-learning mitigations; (D7) McNemar's test for models that score almost the same; (D8) the full
system of Class 04_01 + 04_02: static first, sandbox only when the static model is unsure.
Data: MalBehavD-V1 traces (real Cuckoo), PE headers of Class 04_01, same split as notebook 03.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")

import sys
import warnings

import numpy as np

sys.path[:0] = [".", "..", "../../src", "../../../src"]
import aasplot as ap
import malwlib as mw

warnings.filterwarnings("ignore")
ap.setup("04")
import keras
from keras import layers
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

mbd = mw.load_api_traces("mbd")
L = 100
T = [t[:L] for t in mw.traces(mbd)]
y = mbd["malware"].to_numpy()
itr, ite = train_test_split(np.arange(len(T)), test_size=0.3, stratify=y, random_state=0)
ytr, yte = y[itr], y[ite]
ident = lambda t: t  # noqa: E731
bag = CountVectorizer(analyzer=ident)
lr = LogisticRegression(max_iter=3000).fit(bag.fit_transform([T[i] for i in itr]), ytr)
score = lambda toks: lr.predict_proba(bag.transform(toks))[:, 1]  # noqa: E731
te_mal = [T[i] for i in ite if y[i] == 1]
te_ben = [T[i] for i in ite if y[i] == 0]
print(f"test: {len(te_mal)} malware, {len(te_ben)} benign; bag + LR benign FPR {(score(te_ben) > .5).mean():.3f},"
      f" malware recall {(score(te_mal) > .5).mean():.3f}")  # fmt: skip

## D5a. How fast is a sandbox? A throughput model (assumption)

One analysis = restore a clean state + run the sample for a fixed window. The restore times below are orders of
magnitude, not measurements (they depend on hardware and image size); the shape of the curves is the point.

In [ ]:
RESTORE = {"container": 1, "microVM": 1, "VM snapshot": 20, "bare metal": 300}  # seconds (assumption)
windows = np.array([30, 60, 120, 300, 600])
f, ax = ap.fig("full", 1.5)
for (k, r), c in zip(RESTORE.items(), [ap.GREEN, ap.BLUE, ap.GOLD, ap.RED]):
    ax.plot(range(len(windows)), 3600 / (r + windows), "-o", ms=3, color=c, label=f"{k} (restore {r} s)")
ax.set_xticks(range(len(windows)), windows)
ax.set_xlabel("analysis window (s)")
ax.set_ylabel("samples per hour per host slot")
ax.set_yscale("log")
ax.legend(fontsize=6, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.3))
ap.save(f, "d04_sandbox_throughput")
for k, r in RESTORE.items():
    print(f"{k:12s} {3600 / (r + 120):6.1f} samples/hour at a 120 s window; {3600 / (r + 600):6.1f} at 600 s")

## D5. A sandbox-aware sample, seen from the model

Two evasions that leave a trace behind (simulated on real malware traces, nothing is executed):

* **Early exit**: the sample probes the machine, finds a sandbox and stops: only the first $k$ calls exist.
* **Stalling**: the sample burns time in a loop of one harmless-looking call (a timer poll); the sandbox records
  that call again and again until its call budget (here 100) is spent, so the payload never enters the window.
  The filler is the call the model trusts most (most negative weight among calls that occur in benign traces):
  a worst case for the defender. A sleep call itself would not help the attacker: `ntdelayexecution` is a
  malware indicator in this corpus.

Detector: bag of calls + logistic regression, threshold .5, trained on complete traces.

In [ ]:
names_ = np.array(bag.get_feature_names_out())
benign_df = np.asarray(bag.transform([T[i] for i in itr if y[i] == 0]).astype(bool).sum(0)).ravel()
SLEEP = str(names_[np.argmin(np.where(benign_df >= 100, lr.coef_[0], 9))])
print("filler call:", SLEEP, "(weight", round(float(lr.coef_[0][list(names_).index(SLEEP)]), 2), ")")
ks = [100, 40, 20, 10, 5, 3]
rec_k = [(score([t[:k] for t in te_mal]) > 0.5).mean() for k in ks]
print("early exit, malware recall by calls kept:", {k: round(float(v), 3) for k, v in zip(ks, rec_k)})
ss = [0, 20, 50, 80, 95]
rec_s = [(score([([SLEEP] * s + t)[:L] for t in te_mal]) > 0.5).mean() for s in ss]
print("stalling, malware recall by filler calls first:", {k: round(float(v), 3) for k, v in zip(ss, rec_s)})
f, axs = ap.fig("full", 1.5, ncols=2, sharey=True)
axs[0].plot(range(len(ks)), rec_k, "-o", color=ap.RED, ms=3)
axs[0].set_xticks(range(len(ks)), ks)
axs[0].set_xlabel("calls kept (early exit)")
axs[0].set_ylabel("malware recall")
axs[1].plot(range(len(ss)), rec_s, "-o", color=ap.RED, ms=3)
axs[1].set_xticks(range(len(ss)), ss)
axs[1].set_xlabel("filler calls first (stalling)")
for a in axs:
    a.set_ylim(0, 1.02)
ap.save(f, "d04_evasion_recall")

## D6. Mitigations that live in the model

1. **Collapse repeats**: a loop of identical calls becomes one call (defeats stalling).
2. **Prefix augmentation**: train on random prefixes too, so short traces are normal for the model.
3. **Abstain**: a score in a grey band, a trace with fewer than 15 calls, or a trace in which one call makes up
   more than half of all calls is *unknown*, not benign; unknown goes to a second environment or an analyst.
4. **Probe features (a failure case)**: the sample must look before it hides, so count environment-probing calls
   in the first 10. Do they separate the classes?

In [ ]:
rng = np.random.default_rng(0)
tr_tok = [T[i] for i in itr]
aug = [t[: int(rng.integers(3, max(4, len(t) + 1)))] for t in tr_tok for _ in range(2)]
lr_aug = LogisticRegression(max_iter=3000).fit(bag.transform(tr_tok + aug), np.r_[ytr, np.repeat(ytr, 2)])
cbag = CountVectorizer(analyzer=ident)
lr_col = LogisticRegression(max_iter=3000).fit(cbag.fit_transform([mw.collapse_repeats(t) for t in tr_tok]), ytr)
cs = lambda toks: lr_col.predict_proba(cbag.transform([mw.collapse_repeats(t) for t in toks]))[:, 1]  # noqa: E731
PROBES = {"getcursorpos", "isdebuggerpresent", "ntquerysysteminformation", "getsysteminfo", "getnativesysteminfo",
          "getsystemmetrics", "getkeystate", "findwindoww", "getcomputernamew", "getusernamew", "getdiskfreespaceexw",
          "ntdelayexecution", "getsystemtimeasfiletime", "createtoolhelp32snapshot"}  # fmt: skip
sets = {
    "complete trace": (te_mal, te_ben),
    "early exit (10 calls)": ([t[:10] for t in te_mal], te_ben),
    "stalling (80 filler calls)": ([([SLEEP] * 80 + t)[:L] for t in te_mal], te_ben),
}
models = {
    "plain": lambda toks: score(toks),
    "+ collapse repeats": cs,
    "+ prefix augmentation": lambda toks: lr_aug.predict_proba(bag.transform(toks))[:, 1],
}
rows = []
for sname, (m_, b_) in sets.items():
    for mname, fn in models.items():
        rows.append((sname, mname, (fn(m_) > 0.5).mean(), (fn(b_) > 0.5).mean()))
        print(f"{sname:22s} {mname:22s} recall {rows[-1][2]:.3f}  benign FPR {rows[-1][3]:.3f}")
f, ax = ap.fig("full", 1.5)
w_ = 0.27
for j, mname in enumerate(models):
    v = [r[2] for r in rows if r[1] == mname]
    ax.bar(np.arange(len(sets)) + (j - 1) * w_, v, w_, color=[ap.GREY, ap.BLUE, ap.GREEN][j], label=mname)
ax.set_xticks(range(len(sets)), list(sets))
ax.set_ylabel("malware recall")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=6, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ap.save(f, "d04_mitigation_recall")

In [ ]:
# abstain: unknown when the trace is quiet (< 15 calls), dominated by one call, or the score is in the grey band
def decide(toks, fn, lo=0.3, hi=0.7, min_calls=15):
    p = fn(toks)
    dom = np.array([max(np.unique(t, return_counts=True)[1]) / len(t) for t in toks])
    unk = (np.array([len(t) for t in toks]) < min_calls) | (dom > 0.5) | ((p > lo) & (p < hi))
    return np.where(unk, 2, (p > 0.5).astype(int))  # 0 benign, 1 malware, 2 unknown


print("decision shares (benign / malware / unknown):")
for sname, (m_, b_) in sets.items():
    dm, db = decide(m_, models["+ collapse repeats"]), decide(b_, models["+ collapse repeats"])
    print(f"{sname:22s} malware -> {[round(float((dm == k).mean()), 3) for k in range(3)]}"
          f"   benign -> {[round(float((db == k).mean()), 3) for k in range(3)]}")  # fmt: skip
f, ax = ap.fig("col", 1.9)
labs = ["benign", "malware", "unknown"]
cols = [ap.BLUE, ap.RED, ap.GOLD]
for j, sname in enumerate(sets):
    dm = decide(sets[sname][0], models["+ collapse repeats"])
    left = 0
    for k in range(3):
        v = float((dm == k).mean())
        ax.barh(j, v, left=left, color=cols[k], label=labs[k] if j == 0 else None)
        left += v
ax.set_yticks(range(len(sets)), ["complete", "early exit", "stalling"])
ax.invert_yaxis()
ax.set_xlabel("what happens to malware")
ax.legend(fontsize=6.5, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.3), columnspacing=0.8, handlelength=1)
ap.save(f, "d04_abstain")

In [ ]:
def probe_hits(t, n=10):
    return len(set(t[:n]) & PROBES)


groups = {"benign": te_ben, "malware": te_mal}
f, ax = ap.fig("col", 1.8)
for j, (g, toks) in enumerate(groups.items()):
    h = np.array([probe_hits(t) for t in toks])
    print(f"{g:8s} mean distinct probe calls in first 10: {h.mean():.2f}; share with >= 2: {(h >= 2).mean():.3f}")
    ax.bar(j, (h >= 2).mean(), 0.6, color=[ap.BLUE, ap.RED][j])
ax.set_xticks([0, 1], list(groups))
ax.set_ylabel("share with >= 2 probes\nin the first 10 calls")
ap.save(f, "d04_probes")

## D7. McNemar's test: are two close models really different?

On the same 771 test traces two classifiers each get some traces right and some wrong. Only the traces on which
they **disagree** carry information: $b$ = A right and B wrong, $c$ = A wrong and B right. Under H0 (equal error
rate) each disagreement is a fair coin, so $\min(b,c)$ is Binomial$(b+c, 1/2)$. With $b+c \ge 25$ the statistic
$(|b-c|-1)^2/(b+c)$ is chi-squared with 1 degree of freedom (Edwards 1948); below 25 use the exact binomial.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

te_tok = [T[i] for i in ite]
tok = {w: i + 1 for i, w in enumerate(sorted({w for t in T for w in t}))}
ids = lambda seqs: np.array([[tok[w] for w in s_] + [0] * (L - len(s_)) for s_ in seqs], np.int32)  # noqa: E731
Xtr, Xte = ids(tr_tok), ids(te_tok)


def fit_gru(seed):
    keras.utils.set_random_seed(seed)
    m_ = keras.Sequential([layers.Input((L,)), layers.Embedding(len(tok) + 1, 32, mask_zero=True), layers.GRU(64),
                           layers.Dense(1, activation="sigmoid")])  # fmt: skip
    m_.compile("adam", "binary_crossentropy")
    m_.fit(Xtr, ytr, epochs=12, batch_size=64, validation_split=0.15, verbose=0)
    return (m_.predict(Xte, verbose=0)[:, 0] > 0.5).astype(int)


A_tr, A_te = bag.transform(tr_tok), bag.transform(te_tok)
ng_an = lambda t: (
    t + [a + ">" + b for a, b in zip(t, t[1:])] + [a + ">" + b + ">" + c for a, b, c in zip(t, t[1:], t[2:])]
)  # noqa: E731
ngv = TfidfVectorizer(analyzer=ng_an, min_df=3, sublinear_tf=True)
N_tr = ngv.fit_transform(tr_tok)
pred = {
    "bag+LR": lr.predict(A_te),
    "n-gram+LR": LogisticRegression(max_iter=3000).fit(N_tr, ytr).predict(ngv.transform(te_tok)),
    "bag+RF": RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(A_tr, ytr).predict(A_te),
    "bag+GBM": HistGradientBoostingClassifier(random_state=0).fit(A_tr.toarray(), ytr).predict(A_te.toarray()),
    "GRU s0": fit_gru(0),
    "GRU s1": fit_gru(1),
}
for k, p in pred.items():
    print(f"{k:10s} accuracy {(p == yte).mean():.4f}  errors {(p != yte).sum()}")
names = list(pred)
Pm = np.ones((len(names), len(names)))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i != j:
            Pm[i, j] = mw.mcnemar(yte, pred[a], pred[b])["p_exact"]
for a, b in (("bag+LR", "bag+RF"), ("bag+LR", "n-gram+LR"), ("GRU s0", "GRU s1"), ("bag+RF", "GRU s0")):
    r = mw.mcnemar(yte, pred[a], pred[b])
    print(f"{a} vs {b}: b={r['b']} c={r['c']} chi2={r['chi2']:.2f} p(chi2)={r['p_chi2']:.3f} p(exact)={r['p_exact']:.3f}")  # fmt: skip

In [ ]:
f, ax = ap.fig("col", 2.1)
im = ax.imshow(-np.log10(np.clip(Pm, 1e-6, 1)), cmap="Blues", vmin=0, vmax=3)
ax.set_xticks(range(len(names)), names, rotation=60, ha="right", fontsize=6.5)
ax.set_yticks(range(len(names)), names, fontsize=6.5)
for (i, j), v in np.ndenumerate(Pm):
    if i != j:
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6, color="white" if -np.log10(max(v, 1e-6)) > 1.6 else "black")  # fmt: skip
ax.grid(False)
ap.save(f, "d04_mcnemar_matrix")
r = mw.mcnemar(yte, pred["bag+LR"], pred["bag+RF"])
f, ax = ap.fig("col", 1.8)
ax.bar([0, 1], [r["b"], r["c"]], 0.6, color=[ap.BLUE, ap.GREEN])
ax.set_xticks([0, 1], ["A right,\nB wrong ($b$)", "A wrong,\nB right ($c$)"])
ax.set_ylabel("traces (A = bag+LR, B = bag+RF)")
ax.set_title(f"exact p = {r['p_exact']:.3f}", fontsize=7.5)
ap.save(f, "d04_mcnemar_pair")

## D8. The full system: static first, sandbox when unsure

Class 04_01 ended with two static stages (binary, then family). Class 04_02 adds the sandbox and the same two
stages on behaviour:

1. Static binary model (PE headers, notebook 01) scores the file in milliseconds.
2. Confident benign or confident malware: decide now (static family model for a malware verdict).
3. Grey band: run the sandbox, score the trace with the dynamic binary model; if malware, the dynamic family model.

The static (PE) and dynamic (trace) corpora hold *different files*, so each simulated file pairs a PE row with a
trace of the same true class. This assumes the two models' errors are independent; on real files the hard
cases are shared, so the saving below is an upper bound.

In [ ]:
Xpe, ype, _ = mw.load_pe()
Xa, Xb, ya, yb = train_test_split(Xpe, ype, test_size=0.3, stratify=ype, random_state=0)
rf = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(Xa, ya)
ps = rf.predict_proba(Xb)[:, 1]
print(f"static PE model: AUC {roc_auc_score(yb, ps):.4f} (benign share of test {1 - yb.mean():.2f})")
pdyn_te = score(te_tok)  # dynamic binary model, probability of malware, per test trace
rng = np.random.default_rng(5)
pool = {k: np.flatnonzero(yte == k) for k in (0, 1)}
# realistic queue: 10% malware, stratified resample of PE test files and one paired trace per file
n_total, b = 20000, 0.10
sel = np.r_[
    rng.choice(np.flatnonzero(yb == 1), int(b * n_total)),
    rng.choice(np.flatnonzero(yb == 0), n_total - int(b * n_total)),
]
yq, psq = yb[sel], ps[sel]
pdq = np.array([pdyn_te[rng.choice(pool[k])] for k in yq])
SEC_STATIC, SEC_SANDBOX = 0.01, 120.0  # (assumption) seconds per file, order of magnitude of a Cuckoo run
rows = []
for half in np.linspace(0, 0.5, 26):
    esc = (psq > 0.5 - half) & (psq < 0.5 + half) if half > 0 else np.zeros(len(yq), bool)
    pred_c = np.where(esc, pdq > 0.5, psq > 0.5)
    tp, fp, fn = ((pred_c == 1) & (yq == 1)).sum(), ((pred_c == 1) & (yq == 0)).sum(), ((pred_c == 0) & (yq == 1)).sum()
    rows.append(
        (
            half,
            esc.mean(),
            f1_score(yq, pred_c),
            tp / (tp + fn),
            fp / (yq == 0).sum(),
            SEC_STATIC + esc.mean() * SEC_SANDBOX,
        )
    )
R = np.array(rows)
for half in (0.0, 0.1, 0.2, 0.5):
    r = R[np.argmin(abs(R[:, 0] - half))]
    print(f"band +-{r[0]:.2f}: sandbox share {r[1]:.3f}  F1 {r[2]:.3f}  recall {r[3]:.3f}  FPR {r[4]:.4f} ({100 * r[4]:.1f}%)  mean cost {r[5]:.2f} s/file")  # fmt: skip
all_dyn = (pdq > 0.5).astype(int)
print(
    f"static only: F1 {R[0, 2]:.3f} | sandbox for every file: F1 {f1_score(yq, all_dyn):.3f} at {SEC_SANDBOX:.0f} s/file"
)
f, ax = ap.fig("full", 1.3, ncols=2)
ax[0].plot(100 * R[:, 1], R[:, 2], "-o", color=ap.BLUE, ms=2.5)
ax[0].set_xlabel("files sent to the sandbox (%)")
ax[0].set_ylabel("F1 (queue with 10% malware)")
ax[1].plot(100 * R[:, 1], 100 * R[:, 4], "-o", color=ap.RED, ms=2.5, label="false alarms (% of benign)")
ax[1].plot(100 * R[:, 1], 100 * (1 - R[:, 3]), "-o", color=ap.GOLD, ms=2.5, label="missed malware (%)")
ax[1].set_xlabel("files sent to the sandbox (%)")
ax[1].set_ylabel("error (%)")
ax[1].legend(fontsize=6)
ap.save(f, "d04_cascade")

In [ ]:
# family stages: each stage has its own measured quality (dynamic here, static in notebook 02)
FT = mw.traces(mw.load_api_traces("mal2019"), 200)
fy = mw.load_api_traces("mal2019")["family"].to_numpy()
fa, fb = train_test_split(np.arange(len(FT)), test_size=0.3, stratify=fy, random_state=0)
fv = TfidfVectorizer(analyzer=ng_an, min_df=3, sublinear_tf=True, max_features=40000)
fm = LogisticRegression(max_iter=2000).fit(fv.fit_transform([FT[i] for i in fa]), fy[fa])
fp_ = fm.predict(fv.transform([FT[i] for i in fb]))
print(f"dynamic family model (8 families, n-gram + LR): macro-F1 {f1_score(fy[fb], fp_, average='macro'):.3f}, accuracy {(fp_ == fy[fb]).mean():.3f}")  # fmt: skip
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(fy[fb], fp_, labels=sorted(set(fy)), normalize="true")
f, ax = ap.fig("col", 2.2)
ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(8), sorted(set(fy)), rotation=60, ha="right", fontsize=5.5)
ax.set_yticks(range(8), sorted(set(fy)), fontsize=5.5)
ax.set_xlabel("predicted")
ax.grid(False)
ap.save(f, "d04_family_confusion")

## Takeaways

- Early exit and stalling both fold the plain detector; collapsing repeats and prefix augmentation recover most
  of the recall, and an *unknown* verdict keeps a quiet trace from being called benign.
- Close models are compared with McNemar on the same test traces, not by eyeballing accuracy.
- The cascade sends only the grey band to the sandbox: most of the F1 for a small share of the sandbox time.